# Lab: Ensemble Methods, Boosting and Bagging

**Master Smart Data, ENSAI** · Duration: 2 hours

A single decision tree is easy to understand but rarely accurate. Ensemble methods train **many trees** and **combine** their predictions. There are two main families.

* **Boosting** trains simple trees **one after the other**. Each new tree corrects the errors of the previous ones.
* **Bagging** trains deep trees **independently** on random versions of the data, then lets them vote. Random forests belong to this family.

Both families are built on decision trees, so we first recall how a single tree works.

| Part | Topic | Data | Time |
|---|---|---|---|
| 0 | Environment check and data | | 5 min |
| 1 | The decision tree | moons, Cover Type | 15 min |
| 2 | Boosting: AdaBoost, gradient boosting, XGBoost | moons, Cover Type, a 1D signal | 50 min |
| 3 | Bagging and random forests | moons, Cover Type | 40 min |
| 4 | Synthesis | Cover Type (two classes) | 10 min |

Before starting, you must have installed the environment by following the `README.md` of the repository.

**How to read this notebook**

* 💻 **Coding task**: complete the function or the line marked `# TODO`. Read the docstring first: it gives the inputs and the outputs.
* ✅ **Test cell**: run it right after your code. It prints `OK` when your implementation is correct.
* 💬 **Question**: write a short answer (two or three sentences) in the cell below it.
* 🎁 **Bonus**: skip it if you are short on time, and come back to it at the end.

**The thread of the lab.** A shallow tree is too simple and a deep tree is unstable. Boosting fixes the first problem, bagging fixes the second.

---
## Part 0. Environment check and data

Run the cell below first. It checks that this notebook runs inside the virtual environment you created, and that the required packages are installed. If it raises an error, go back to the `README.md` (section *Troubleshooting*).

In [ ]:
import sys
from importlib.metadata import version, PackageNotFoundError

print("Python      :", sys.version.split()[0])
print("Interpreter :", sys.executable)

if sys.prefix == sys.base_prefix:
    raise RuntimeError(
        "This notebook is NOT running inside a virtual environment.\n"
        "Activate .venv in your terminal, then restart Jupyter from that terminal "
        "(or select the .venv interpreter in VS Code)."
    )

missing = []
for package in ["numpy", "matplotlib", "scikit-learn", "pandas", "xgboost"]:
    try:
        print("%-12s: %s" % (package, version(package)))
    except PackageNotFoundError:
        missing.append(package)
if missing:
    raise RuntimeError("Missing packages: %s. Run: pip install -r requirements.txt" % ", ".join(missing))

print("\nEnvironment OK")

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingRegressor, RandomForestClassifier
from xgboost import XGBClassifier

SEED = 0
plt.rcParams["figure.dpi"] = 100

### The toy dataset

Two interleaved half moons with noise. Labels are in $\{-1, 1\}$. In 2D we can draw the decision boundaries and *see* what each algorithm does.

In [ ]:
Xm, ym = make_moons(n_samples=300, noise=0.35, random_state=SEED)
ym = 2 * ym - 1  # labels {0, 1} -> {-1, 1}
Xm_train, Xm_test, ym_train, ym_test = train_test_split(
    Xm, ym, test_size=0.5, random_state=SEED
)


def plot_boundary(predict, X, y, ax, title="", sizes=None):
    '''Plot the decision regions of a binary classifier in 2D.

    Parameters
    ----------
    predict : callable
        Function mapping an array of shape (n, 2) to labels in {-1, 1}.
    X : array of shape (n, 2)
        Points to display.
    y : array of shape (n,)
        Labels of the points, in {-1, 1}.
    ax : matplotlib axis
        Where to draw.
    title : str
        Title of the plot.
    sizes : array of shape (n,) or None
        Marker sizes (used to display the AdaBoost weights).
    '''
    x0_min, x0_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    x1_min, x1_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x0_min, x0_max, 300),
                         np.linspace(x1_min, x1_max, 300))
    Z = predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=[-2, 0, 2], colors=["#9ecae1", "#fdae6b"], alpha=0.6)
    colors = np.where(y == 1, "#e6550d", "#3182bd")
    s = 15 if sizes is None else sizes
    ax.scatter(X[:, 0], X[:, 1], c=colors, s=s, edgecolor="k", linewidth=0.3)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])


fig, ax = plt.subplots(figsize=(5, 4))
plot_boundary(lambda Z: np.zeros(len(Z)), Xm_train, ym_train, ax, "Training points")
plt.show()

### The Forest Cover Type dataset

Each row is a $30 \times 30$ m patch of forest in Colorado. The goal is to predict the **dominant tree species** (7 classes) from 54 variables: elevation, slope, distances to water and roads, sunlight, and indicator variables for the wilderness area and the soil type. The 7 classes are balanced. We use it to measure performance on real data.

The `Id` column is just a row number. We drop it.

In [ ]:
data = pd.read_csv("data/covertype.csv").drop(columns="Id")
print("Size of the data:", data.shape)
data.head()

In [ ]:
cover_names = {1: "Spruce/Fir", 2: "Lodgepole Pine", 3: "Ponderosa Pine",
               4: "Cottonwood/Willow", 5: "Aspen", 6: "Douglas-fir", 7: "Krummholz"}
print(data["Cover_Type"].map(cover_names).value_counts())

feature_names = data.drop(columns="Cover_Type").columns
X = data.drop(columns="Cover_Type").values
y = data["Cover_Type"].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)
print("Train:", X_train.shape, " Test:", X_test.shape)

---
## Part 1. The decision tree (15 min)

### 1.1 Principle

A decision tree classifies a point by asking a **sequence of simple questions**, each of the form
$$\text{"is variable } j \text{ smaller than threshold } s \text{ ?"}$$

* Training starts with all the points in one node (the root).
* It looks for the question $(j, s)$ that best separates the classes into two groups. "Best" means that each group is as **pure** as possible, i.e. dominated by one class. Purity is measured by the *Gini index* or the *entropy*.
* It splits the node in two and repeats on each child, until a stopping rule (for instance a maximal depth).
* Each final node (a **leaf**) predicts the majority class of its training points.

Geometrically, a tree cuts the space into **rectangles** with sides parallel to the axes.

The main hyperparameter is the **depth** of the tree. A tree of depth 1, called a **stump**, asks a single question.

In [ ]:
stump = DecisionTreeClassifier(max_depth=1, random_state=SEED).fit(Xm_train, ym_train)
plt.figure(figsize=(5, 3))
plot_tree(stump, feature_names=["x0", "x1"], filled=True)
plt.show()

### 1.2 💻 Coding task: depth and decision boundary

Train trees of depth 1, 3 and 15 on the moons and display their decision boundaries.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, depth in zip(axes, [1, 3, 15]):
    # TODO: create a DecisionTreeClassifier of depth `depth` (random_state=SEED)
    # and fit it on (Xm_train, ym_train)
    tree = ...
    acc = accuracy_score(ym_test, tree.predict(Xm_test))
    plot_boundary(tree.predict, Xm_train, ym_train, ax, f"depth = {depth}, test accuracy = {acc:.2f}")
plt.show()

💬 **Question 1.1.** Describe the three boundaries. Which tree is too simple? Which one learns the noise of the training set?

*Your answer:*

### 1.3 💻 Coding task: depth and error on real data

Compute the training and test errors of a tree on Cover Type for several depths.

In [ ]:
depths = [1, 2, 4, 6, 8, 10, 15, 20, 30]
train_errors, test_errors = [], []
for depth in depths:
    # TODO: fit a tree of depth `depth` on (X_train, y_train), then append its
    # train error and test error (error = 1 - accuracy, see the .score method)
    ...

plt.plot(depths, train_errors, "o-", label="train error")
plt.plot(depths, test_errors, "o-", label="test error")
plt.xlabel("tree depth")
plt.ylabel("error")
plt.legend()
plt.show()
print(f"Best test error of a single tree: {min(test_errors):.3f}")

💬 **Question 1.2.** How do the two errors evolve with the depth? What is the best test error of a single tree?

*Your answer:*

**Take-away.** A tree is either too simple or unstable. The two parts of this lab fix each problem.
* **Boosting** combines many **simple** trees to build a rich model.
* **Bagging** averages many **deep** trees to make them stable.

---
## Part 2. Boosting (50 min)

### 2.1 AdaBoost: principle

AdaBoost (*Adaptive Boosting*, Freund and Schapire, 1997) works for binary labels $y_i \in \{-1, 1\}$. It trains $T$ simple trees $h_1, \dots, h_T$ **one after the other**. Each point $x_i$ of the training set has a weight $w_i$.

* At the start, all points have the same weight.
* After each tree, the misclassified points get a **larger weight**. The next tree is trained on the weighted data, so it focuses on these difficult points.
* Each tree also receives a vote weight $\alpha_t$. An accurate tree gets a large $\alpha_t$.
* The final prediction is a **weighted vote** of all the trees.

**Weak learners.** AdaBoost is designed for *weak learners*: models that are only slightly better than random guessing, with an error a bit below 50 %. The classical choice is a **stump** (a tree of depth 1) or a very shallow tree. Boosting turns many weak learners into a strong one. Section 2.5 shows why deep trees are a bad choice.

**Algorithm.**

1. Initialise $w_i = 1/n$ for all $i$.
2. For $t = 1, \dots, T$:
    1. Fit a shallow tree $h_t$ on the data with weights $w$.
    2. Compute its weighted error: $\displaystyle \varepsilon_t = \frac{\sum_{i : h_t(x_i) \neq y_i} w_i}{\sum_i w_i}$.
    3. Compute its vote weight: $\displaystyle \alpha_t = \log \frac{1 - \varepsilon_t}{\varepsilon_t}$.
    4. Multiply the weight of each misclassified point by $e^{\alpha_t}$, then normalise the weights so that they sum to 1.
3. Predict with the sign of the weighted vote: $\displaystyle \hat{y}(x) = \mathrm{sign}\Big(\sum_{t=1}^T \alpha_t h_t(x)\Big)$.

A tree with error $\varepsilon_t = 0.5$ does no better than chance. It gets $\alpha_t = 0$ and has no influence. The smaller $\varepsilon_t$, the larger $\alpha_t$.

In scikit-learn, the weights are passed to a tree with `tree.fit(X, y, sample_weight=w)`.

### 2.2 💻 Coding task: implement AdaBoost

In [ ]:
def adaboost_fit(X, y, n_trees, max_depth=1):
    '''Train AdaBoost with decision trees as weak learners.

    Parameters
    ----------
    X : array of shape (n, p)
        Training inputs.
    y : array of shape (n,)
        Training labels, in {-1, 1}.
    n_trees : int
        Number of trees T.
    max_depth : int
        Depth of each tree (1 means stumps).

    Returns
    -------
    trees : list of DecisionTreeClassifier
        The fitted trees, in training order.
    alphas : list of float
        The vote weight of each tree.
    weights : list of arrays of shape (n,)
        The point weights after each round (for visualisation).
    '''
    n = X.shape[0]
    w = np.ones(n) / n
    trees, alphas, weights = [], [], []
    for t in range(n_trees):
        # TODO 1. create a tree of depth max_depth (random_state=SEED)
        # and fit it on (X, y) with the weights w
        tree = ...

        # TODO 2. boolean array `wrong` of misclassified points, then weighted error `err`
        wrong = ...
        err = ...
        err = np.clip(err, 1e-10, 1 - 1e-10)  # avoids a division by zero if err = 0

        # TODO 3. vote weight
        alpha = ...

        # TODO 4. multiply the weights of misclassified points by exp(alpha), then normalise
        w = ...

        trees.append(tree)
        alphas.append(alpha)
        weights.append(w.copy())
    return trees, alphas, weights


def adaboost_predict(trees, alphas, X):
    '''Predict with AdaBoost.

    Parameters
    ----------
    trees : list of fitted trees
    alphas : list of float
        Vote weights of the trees.
    X : array of shape (m, p)

    Returns
    -------
    y_pred : array of shape (m,), labels in {-1, 1}
    '''
    # TODO: weighted vote sum_t alpha_t * h_t(X), then its sign (use 1 when the score is 0)
    ...

In [ ]:
# ✅ Test cell: scikit-learn's AdaBoost follows the same algorithm in the binary case
_trees, _alphas, _weights = adaboost_fit(Xm_train, ym_train, n_trees=20)
_ref = AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=20,
                          random_state=SEED).fit(Xm_train, ym_train)
assert len(_trees) == 20 and len(_alphas) == 20
assert np.isclose(np.sum(_weights[-1]), 1), "the weights must sum to 1"
assert np.allclose(_alphas, _ref.estimator_weights_), "the vote weights alpha are not correct"
assert np.array_equal(adaboost_predict(_trees, _alphas, Xm_test), _ref.predict(Xm_test)), \
    "adaboost_predict is not correct"
print("adaboost_fit and adaboost_predict: OK")

### 2.3 AdaBoost in action

We train 50 stumps on the moons. We display the boundary of the vote of the first 1, 5 and 50 stumps. The size of each point is proportional to its weight at that step.

In [ ]:
trees, alphas, weights = adaboost_fit(Xm_train, ym_train, n_trees=50, max_depth=1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, k in zip(axes, [1, 5, 50]):
    predict_k = lambda Z: adaboost_predict(trees[:k], alphas[:k], Z)
    acc = accuracy_score(ym_test, predict_k(Xm_test))
    sizes = 3000 * weights[k - 1]
    plot_boundary(predict_k, Xm_train, ym_train, ax, f"{k} stump(s), test accuracy = {acc:.2f}", sizes)
plt.show()

💬 **Question 2.1.** Which points get large weights? How can a vote of stumps, each of which is a single straight cut, produce such a boundary?

*Your answer:*

### 2.4 💻 Coding task: learning curves on real data

AdaBoost as written above is binary. We keep two classes of Cover Type: *Ponderosa Pine* (class 3, label $+1$) and *Cottonwood/Willow* (class 4, label $-1$).

The function `staged_errors` computes the error of the vote of the first $k$ trees, for every $k$.

In [ ]:
def to_binary(X, y, pos=3, neg=4):
    '''Keep classes pos and neg, with labels +1 and -1.'''
    mask = np.isin(y, [pos, neg])
    return X[mask], np.where(y[mask] == pos, 1, -1)


Xb_train, yb_train = to_binary(X_train, y_train)
Xb_test, yb_test = to_binary(X_test, y_test)
print("Binary train set:", Xb_train.shape, " test set:", Xb_test.shape)


def staged_errors(trees, alphas, X, y):
    '''Error of the vote of the first k trees, for k = 1, ..., T.'''
    score = np.zeros(X.shape[0])
    errors = []
    for tree, alpha in zip(trees, alphas):
        score += alpha * tree.predict(X)
        errors.append(np.mean(np.where(score >= 0, 1, -1) != y))
    return np.array(errors)


best_tree_error = min(
    1 - DecisionTreeClassifier(max_depth=d, random_state=SEED).fit(Xb_train, yb_train).score(Xb_test, yb_test)
    for d in [1, 2, 3, 5, 8, 10, 15, None]
)
print(f"Best test error of a single tree on this problem: {best_tree_error:.3f}")

Train AdaBoost with $T = 300$ trees, first with stumps (depth 1), then with trees of depth 3. Plot the train and test errors as a function of the number of trees.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, depth in zip(axes, [1, 3]):
    # TODO: train AdaBoost with 300 trees of depth `depth` on (Xb_train, yb_train)
    trees, alphas, _ = ...
    # TODO: plot the train and test errors with staged_errors
    ...
    ax.axhline(best_tree_error, color="gray", ls="--", label="best single tree (test)")
    ax.set_title(f"AdaBoost, trees of depth {depth}")
    ax.set_xlabel("number of trees")
    ax.legend()
axes[0].set_ylabel("error")
plt.show()

💬 **Question 2.2.** Compare with the best single tree. What is the effect of the depth of the trees? Once the train error reaches 0, does the test error stop improving?

*Your answer:*

### 2.5 Why weak learners?

What happens if we give AdaBoost fully grown trees? Run the cell below. It uses 20 trees with no depth limit.

In [ ]:
trees, alphas, weights = adaboost_fit(Xb_train, yb_train, n_trees=20, max_depth=None)
print("vote weights of the first 5 trees:", np.round(alphas[:5], 1))
print("training error of the first 5 trees:", [float(np.mean(t.predict(Xb_train) != yb_train)) for t in trees[:5]])
print(f"largest change of a point weight after 20 rounds: {np.max(np.abs(weights[-1] - 1 / len(yb_train))):.1e}")
print(f"AdaBoost with 20 deep trees, test error: {staged_errors(trees, alphas, Xb_test, yb_test)[-1]:.3f}")
single = DecisionTreeClassifier(random_state=SEED).fit(Xb_train, yb_train)
print(f"one deep tree, test error:                {1 - single.score(Xb_test, yb_test):.3f}")

💬 **Question 2.3.** Look at the training errors, the vote weights and the point weights. Why does AdaBoost behave like a single deep tree here? Why does boosting need weak learners?

*Your answer:*

### 2.6 Gradient boosting: principle

Gradient boosting (Friedman, 2001) keeps the idea of trees trained one after the other, each one correcting the previous ones. It describes the correction in a more direct way. We present it in **regression**, where it is very intuitive.

Let $F$ be the current model. Its errors on the training set are the **residuals** $r_i = y_i - F(x_i)$. The next tree learns to predict these residuals. We then add it to the model:

1. Start from a constant: $F_0(x) = \bar{y}$ (mean of the targets).
2. For $t = 1, \dots, T$:
    1. Compute the residuals $r_i = y_i - F_{t-1}(x_i)$.
    2. Fit a regression tree $h_t$ on the pairs $(x_i, r_i)$.
    3. Update: $F_t(x) = F_{t-1}(x) + \eta \, h_t(x)$.

The **learning rate** $\eta \in (0, 1]$ makes each correction small, so that no single tree takes too much importance.

*Why "gradient"?* For the squared loss $\frac12 (y - F)^2$, the residual $y - F$ is exactly minus the gradient of the loss with respect to $F$. Each tree is therefore a step of **gradient descent**. For other losses (for instance in classification), the residuals are simply replaced by minus the gradient of that loss. This is how the method extends to classification.

### 2.7 💻 Coding task: gradient boosting by hand

We use a noisy sine curve in 1D.

In [ ]:
rng = np.random.default_rng(SEED)
x1 = np.sort(rng.uniform(0, 6, 100))
y1 = np.sin(x1) + 0.3 * rng.normal(size=100)
X1 = x1.reshape(-1, 1)
x_grid = np.linspace(0, 6, 500).reshape(-1, 1)

In [ ]:
def gradient_boosting_fit(X, y, n_trees, learning_rate, max_depth=2):
    '''Gradient boosting for regression with the squared loss.

    Parameters
    ----------
    X : array of shape (n, p)
    y : array of shape (n,)
    n_trees : int
        Number of trees T.
    learning_rate : float
        Step eta multiplying each tree.
    max_depth : int
        Depth of the regression trees.

    Returns
    -------
    f0 : float
        Initial constant model.
    trees : list of DecisionTreeRegressor
    '''
    f0 = np.mean(y)
    F = np.full(len(y), f0)  # current predictions on the training set
    trees = []
    for t in range(n_trees):
        # TODO: compute the residuals, fit a DecisionTreeRegressor of depth max_depth
        # (random_state=SEED) on them, update F, and store the tree in `trees`
        ...
    return f0, trees


def gradient_boosting_predict(f0, trees, learning_rate, X):
    '''Prediction F(x) = f0 + eta * sum_t h_t(x).'''
    return f0 + learning_rate * sum(tree.predict(X) for tree in trees)

In [ ]:
# ✅ Test cell: scikit-learn's gradient boosting follows the same algorithm
_f0, _trees = gradient_boosting_fit(X1, y1, n_trees=30, learning_rate=0.1)
_ref = GradientBoostingRegressor(n_estimators=30, learning_rate=0.1, max_depth=2,
                                 criterion="squared_error", random_state=SEED).fit(X1, y1)
assert np.isclose(_f0, np.mean(y1)), "f0 must be the mean of y"
assert len(_trees) == 30
assert np.allclose(gradient_boosting_predict(_f0, _trees, 0.1, x_grid), _ref.predict(x_grid))
print("gradient_boosting_fit: OK")

In [ ]:
def show_gradient_boosting(learning_rate, n_list=(1, 10, 100)):
    '''Plot the gradient boosting fit after several numbers of trees.'''
    f0, trees = gradient_boosting_fit(X1, y1, n_trees=max(n_list), learning_rate=learning_rate)
    fig, axes = plt.subplots(1, len(n_list), figsize=(15, 3.5), sharey=True)
    for ax, k in zip(axes, n_list):
        ax.scatter(x1, y1, s=10, color="gray")
        ax.plot(x_grid, np.sin(x_grid), "k--", lw=1, label="true function")
        ax.plot(x_grid, gradient_boosting_predict(f0, trees[:k], learning_rate, x_grid), "r", label="model")
        ax.set_title(f"learning rate {learning_rate}, {k} tree(s)")
    axes[0].legend()
    plt.show()


show_gradient_boosting(learning_rate=0.1)

Now run the same experiment with a learning rate of 1.

In [ ]:
# TODO
...

💬 **Question 2.4.** Describe how the model is built tree after tree. What changes with a learning rate of 1?

*Your answer:*

### 2.8 XGBoost

XGBoost (Chen and Guestrin, 2016) is an implementation of gradient boosting that is widely used in practice and very often wins tabular data competitions. The principle is the one of Section 2.6. It adds:

* a **penalty** on the complexity of each tree, which limits overfitting,
* **random subsampling** of rows and columns for each tree (optional),
* a very **fast** implementation.

In multiclass classification, the model computes one score per class and turns them into probabilities. At each round, one tree per class corrects these scores in the direction of minus the gradient of the classification loss.

Its main hyperparameters are `n_estimators` (number of rounds $T$), `learning_rate` ($\eta$) and `max_depth`.

XGBoost needs labels $0, \dots, K-1$, so we shift the classes. We also cut a **validation set** from the training set. It is used to monitor the training without touching the test set.

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train - 1, test_size=0.2, stratify=y_train, random_state=SEED
)

#### 💻 Coding task: learning rate and number of trees

Train XGBoost with 500 trees of depth 6 and learning rates 0.3 and 0.05. We follow two quantities on the validation set after each tree:
* the classification error (`merror`),
* the *log-loss* (`mlogloss`), i.e. the average of $-\log(\text{probability given to the true class})$. It is small when the model gives high probability to the right class and strongly penalises confident mistakes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for lr in [0.3, 0.05]:
    # TODO: create an XGBClassifier with n_estimators=500, learning_rate=lr, max_depth=6,
    # eval_metric=["merror", "mlogloss"], random_state=SEED, n_jobs=-1
    xgb = ...
    xgb.fit(X_tr, y_tr, eval_set=[(X_tr, y_tr), (X_val, y_val)], verbose=False)
    history = xgb.evals_result()  # validation_0 = train set, validation_1 = validation set
    line, = axes[0].plot(history["validation_1"]["merror"], label=f"validation, lr = {lr}")
    axes[0].plot(history["validation_0"]["merror"], ls=":", color=line.get_color(), label=f"train, lr = {lr}")
    axes[1].plot(history["validation_1"]["mlogloss"], color=line.get_color(), label=f"validation, lr = {lr}")
axes[0].set_ylabel("error")
axes[1].set_ylabel("log-loss")
for ax in axes:
    ax.set_xlabel("number of trees")
    ax.legend()
plt.show()

💬 **Question 2.5.** Compare the two learning rates. With lr = 0.3, what happens to the validation log-loss after about 100 trees? What does it mean?

*Your answer:*

#### Early stopping

Rather than fixing the number of trees in advance, we can stop the training when the validation error has not improved for a given number of rounds. This is **early stopping**. In XGBoost, it is the parameter `early_stopping_rounds`.

In [ ]:
xgb = XGBClassifier(n_estimators=2000, learning_rate=0.1, max_depth=6, early_stopping_rounds=50,
                    eval_metric="merror", random_state=SEED, n_jobs=-1)
xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
print("Number of trees kept:", xgb.best_iteration + 1)
print(f"Test error: {1 - xgb.score(X_test, y_test - 1):.3f}")

**Take-away of Part 2.** Boosting trains weak learners (shallow trees) **one after the other**. Each tree corrects the errors of the current model. This reduces the bias of simple models. Too many trees, or too large a learning rate, can lead to overfitting. The number of trees is chosen on a validation set.

---
## Part 3. Bagging and random forests (40 min)

### 3.1 Bagging: principle

We saw in Part 1 that a deep tree fits its training set very well, but changes a lot when the training set changes (high variance). The idea of **bagging** (*bootstrap aggregating*, Breiman, 1996) is to **average** many deep trees.

We only have one training set. To create several different ones, we use the **bootstrap**: we draw $n$ points **with replacement** from the $n$ training points. Some points appear several times, others not at all.

**Algorithm.**

1. For $b = 1, \dots, B$:
    1. Draw a bootstrap sample of the training set.
    2. Fit a deep tree $h_b$ on it.
2. Predict by **majority vote** of the $B$ trees.

Unlike boosting, the trees are **independent** of each other. They can be trained in parallel, and they all have the same vote weight.

Why does this help? Each tree makes errors, but not the same ones, because each saw a different sample. When we vote, these errors partly cancel out. This is the same reason why the average of $B$ noisy measurements is more precise than a single one.

### 3.2 💻 Coding task: implement bagging

In [ ]:
def bagging_fit(X, y, n_trees):
    '''Bagging of fully grown decision trees.

    Parameters
    ----------
    X : array of shape (n, p)
    y : array of shape (n,)
    n_trees : int
        Number of trees B.

    Returns
    -------
    trees : list of DecisionTreeClassifier
    '''
    rng = np.random.default_rng(SEED)
    n = X.shape[0]
    trees = []
    for b in range(n_trees):
        # TODO: draw n indices with replacement (rng.integers), fit a fully grown
        # DecisionTreeClassifier (random_state=SEED) on this bootstrap sample, store it
        ...
    return trees


def bagging_predict(trees, X):
    '''Majority vote of the trees, for labels in {-1, 1}.'''
    # TODO: average of the predictions of the trees, then its sign (use 1 when the average is 0)
    ...

In [ ]:
# ✅ Test cell
_trees = bagging_fit(Xm_train, ym_train, n_trees=5)
assert len(_trees) == 5
assert all(t.get_params()["max_depth"] is None for t in _trees), "the trees must be fully grown (no max_depth)"
assert not np.array_equal(_trees[0].predict(Xm_test), _trees[1].predict(Xm_test)), \
    "the trees must be trained on different bootstrap samples"
_votes = np.mean([t.predict(Xm_test) for t in _trees], axis=0)
assert np.array_equal(bagging_predict(_trees, Xm_test), np.where(_votes >= 0, 1, -1)), \
    "bagging_predict is not correct"
print("bagging_fit and bagging_predict: OK")

### 3.3 Bagging in action

We display three of the bagged trees, then the vote of 100 trees.

In [ ]:
trees = bagging_fit(Xm_train, ym_train, n_trees=100)

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for b in range(3):
    acc = accuracy_score(ym_test, trees[b].predict(Xm_test))
    plot_boundary(trees[b].predict, Xm_train, ym_train, axes[b], f"tree {b + 1}, test accuracy = {acc:.2f}")
acc = accuracy_score(ym_test, bagging_predict(trees, Xm_test))
plot_boundary(lambda Z: bagging_predict(trees, Z), Xm_train, ym_train, axes[3],
              f"vote of 100 trees, test accuracy = {acc:.2f}")
plt.show()

💬 **Question 3.1.** Compare the three individual trees with each other, then with the vote.

*Your answer:*

### 3.4 Random forests: principle

Bagging works best when the trees make **different** errors. But bagged trees often look alike. If one variable is very informative (here, the elevation), almost every tree uses it for its first questions, and the trees end up similar.

A **random forest** (Breiman, 2001) is bagging with one more source of randomness. At **each node**, the tree only looks for the best question among a **random subset of the variables** (often $\sqrt{p}$ of them). The trees become more different from each other. Each tree is a little less accurate, but the vote benefits from the diversity. This is also much faster, since each node examines fewer variables.

In scikit-learn, `RandomForestClassifier` has a parameter `max_features`, the number of variables examined at each node:
* `max_features=None` examines all variables. This is exactly **bagging**.
* `max_features="sqrt"` examines $\sqrt{p}$ variables. This is the usual **random forest**.

### 3.5 💻 Coding task: bagging versus random forest on real data

We come back to the 7 classes of Cover Type. The function below computes the test error of the vote of the first $k$ trees, for every $k$. It also measures the average error of an individual tree, and how often two trees agree.

In [ ]:
def forest_diagnostics(forest, X, y):
    '''Test error of the first k trees, error of a single tree, agreement between trees.

    Returns
    -------
    errors : array of shape (n_trees,)
        Error of the vote of the first k trees, for k = 1, ..., n_trees.
    single_error : float
        Average error of an individual tree.
    agreement : float
        Average proportion of points on which two trees predict the same class
        (computed on the first 20 trees).
    '''
    probas = np.zeros((X.shape[0], len(forest.classes_)))
    errors, preds = [], []
    for tree in forest.estimators_:
        p = tree.predict_proba(X)
        probas += p
        errors.append(np.mean(forest.classes_[np.argmax(probas, axis=1)] != y))
        preds.append(forest.classes_[np.argmax(p, axis=1)])
    single_error = np.mean([np.mean(pred != y) for pred in preds])
    agreement = np.mean([np.mean(preds[i] == preds[j]) for i in range(20) for j in range(i + 1, 20)])
    return np.array(errors), single_error, agreement

Train two forests of 200 trees on `(X_train, y_train)`: one with `max_features=None` (bagging) and one with `max_features="sqrt"` (random forest). Use `n_jobs=-1` to train the trees in parallel, `random_state=SEED`, and `oob_score=True` (used in the bonus).

In [ ]:
models = {
    # TODO: bagging = RandomForestClassifier with max_features=None
    "bagging": ...,
    # TODO: random forest = RandomForestClassifier with max_features="sqrt"
    "random forest": ...,
}

for name, model in models.items():
    start = time.time()
    model.fit(X_train, y_train)
    duration = time.time() - start
    errors, single_error, agreement = forest_diagnostics(model, X_test, y_test)
    plt.plot(np.arange(1, len(errors) + 1), errors, label=name)
    print(f"{name:14s} training time {duration:5.1f} s | error of one tree {single_error:.3f} | "
          f"agreement between trees {agreement:.3f} | error of the vote {errors[-1]:.3f}")

plt.axhline(min(test_errors), color="gray", ls="--", label="best single tree (Part 1)")
plt.xscale("log")
plt.xlabel("number of trees")
plt.ylabel("test error")
plt.legend()
plt.show()

💬 **Question 3.2.** How does the error evolve with the number of trees? Does it go back up? Compare bagging and the random forest: error of one tree, agreement, error of the vote, training time.

*Your answer:*

### 3.6 🎁 Bonus: two by-products of random forests

**Out-of-bag error.** Each bootstrap sample leaves out about 37 % of the points. Indeed, a given point is not drawn in one draw with probability $1 - 1/n$, hence in none of the $n$ draws with probability $(1 - 1/n)^n \approx e^{-1} \approx 0.37$. Each point can therefore be predicted by the trees that did not see it. This gives an estimate of the test accuracy **without** a separate validation set.

In [ ]:
rf = models["random forest"]
print(f"Out-of-bag accuracy: {rf.oob_score_:.3f}")
print(f"Test accuracy:       {rf.score(X_test, y_test):.3f}")

**Variable importance.** The forest measures how much each variable contributed to purifying the nodes, summed over all trees. This gives a first idea of which variables matter.

In [ ]:
importances = pd.Series(rf.feature_importances_, index=feature_names).sort_values()
importances.tail(10).plot.barh(figsize=(6, 4), title="10 most important variables")
plt.show()

---
## Part 4. Synthesis (10 min)

We compare the four methods on the binary problem of Section 2.4 (Ponderosa Pine against Cottonwood/Willow), because our AdaBoost is binary. The boosting methods use shallow trees, the bagging methods use deep trees. We keep almost default hyperparameters. Run the cell.

*Remark.* AdaBoost has a multiclass version (`AdaBoostClassifier` in scikit-learn), but with weak learners it performs poorly on the 7 classes of this dataset. Gradient boosting handles several classes more naturally. This is one reason why it is more used today.

In [ ]:
results = []


def record(name, y_pred, duration):
    '''Store the test error and the training time of a method.'''
    results.append({"method": name,
                    "test error": round(np.mean(y_pred != yb_test), 3),
                    "training time (s)": round(duration, 1)})


start = time.time()
tree = DecisionTreeClassifier(random_state=SEED).fit(Xb_train, yb_train)
record("single deep tree", tree.predict(Xb_test), time.time() - start)

start = time.time()
trees, alphas, _ = adaboost_fit(Xb_train, yb_train, n_trees=300, max_depth=3)
record("AdaBoost (300 trees of depth 3)", adaboost_predict(trees, alphas, Xb_test), time.time() - start)

start = time.time()
xgb = XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, n_jobs=-1)
xgb.fit(Xb_train, (yb_train + 1) // 2)  # XGBoost needs labels 0 and 1
record("XGBoost (300 trees of depth 6)", 2 * xgb.predict(Xb_test) - 1, time.time() - start)

start = time.time()
forest = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED).fit(Xb_train, yb_train)
record("random forest (200 deep trees)", forest.predict(Xb_test), time.time() - start)

pd.DataFrame(results).set_index("method")

💬 **Question 4.1.** Which methods improve on the single tree, and by how much? Can we conclude that one method is the best?

*Your answer:*

**Summary.**

| | Boosting (AdaBoost, XGBoost) | Bagging, random forest |
|---|---|---|
| Base trees | shallow (weak learners) | deep |
| Construction | sequential, each tree corrects the previous ones | independent, in parallel |
| Combination | weighted sum | vote with equal weights |
| Main effect | reduces the **bias** | reduces the **variance** |
| More trees | can overfit, use a validation set | never hurts, only costs time |
| Key hyperparameters | number of trees, learning rate, depth | number of trees, `max_features` |